# Efecto sobre la nota final: Efecto de compaginar trabajo y estudios
> **Objetivo de investigación** ¿Qué efecto tiene tener un trabajo a tiempo parcial sobre la nota del examen final?
> **Dataset utilizado:** `student_performance_dataset.csv`

In [2]:
import pandas as pd
df = pd.read_csv('student_performance_dataset.csv')

### Descripción del dataset
> Información de 1000 estudiantes, las variables clave van a ser (part_time_job y final_exam_score).
>
> Estado de los datos: limpios y sin nulos

In [3]:
df

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D
...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,Male,2.8,75.4,8.2,Masters,Yes,Yes,No,60.5,70.7,C
996,997,Male,6.7,88.4,7.1,NaN,No,Yes,No,82.2,99.5,A
997,998,Female,2.6,84.5,8.0,High School,Yes,Yes,Yes,65.2,79.2,C
998,999,Female,4.6,85.3,8.1,High School,No,No,Yes,52.2,82.2,B


## Análisis estadístico
>Con el objetivo de evaluar el impacto de tener un trabajo a tiempo parcial sobre la nota final, realizaremos una prueba T de Student para muestras independientes. Así contrastaremos si la diferencia de calificaciones entre los alumnos con y sin trabajo es estadísticamente significativa.
### Estudio de Supuestos Previos

Antes de aplicar el test estadístico, verificamos el comportamiento de los datos:
* **Prueba de Shapiro-Wilk:** Para comprobar si las notas siguen una distribución normal.
* **Prueba de Levene:** Para confirmar la homogeneidad de varianzas entre los que trabajan y los que no.

In [17]:
from scipy import stats

notas_sin_trabajo = df[df['part_time_job'] == 'No']['final_exam_score']
notas_con_trabajo = df[df['part_time_job'] == 'Yes']['final_exam_score']
stat_sin, p_norm_sin = stats.shapiro(notas_sin_trabajo)
stat_con, p_norm_con = stats.shapiro(notas_con_trabajo)

print(f'Normalidad (Sin trabajo) - p-value: {p_norm_sin:.9f}')
print(f'Normalidad (Con trabajo) - p-value: {p_norm_con:.4f}')

stat_lev, p_lev = stats.levene(notas_sin_trabajo, notas_con_trabajo)
print(f'Homogeneidad de Varianzas - p-value: {p_lev}')

Normalidad (Sin trabajo) - p-value: 0.000000001
Normalidad (Con trabajo) - p-value: 0.0008
Homogeneidad de Varianzas - p-value: 0.06552834131438612


### Resultado de las pruebas
* **Prueba de Shapiro-Wilk:** Los p-valores de ambos grupos son menores a 0.05, lo que indica que las notas no siguen una distribución estrictamente normal.
* **Prueba de Levene:** Se asume igualdad de varianzas (homocedasticidad), dado que el p-valor (0.065) es superior a 0.05.

### Conclusiones
>Aunque los datos no sigan la normalidad, el amplio tamaño muestral (1000 observaciones) garantiza la fiabilidad de la prueba paramétrica posterior, debido al Teorema del Límite Central

## Prueba T de Student para muestras independientes
> A continuación, compararemos la media de la nota del examen final entre nuestra muestra de estudiantes con trabajo y la muestra de estudiantes sin trabajo

In [5]:
media = df.groupby('part_time_job')['final_exam_score'].agg(
    total_personas='count',
    media_nota='mean'
).reset_index()

media

,part_time_job,total_personas,media_nota
0,No,684,84.582310
1,Yes,316,81.294937


In [20]:
notas_con_trabajo = df[df['part_time_job'] == 'Yes']['final_exam_score']
notas_sin_trabajo = df[df['part_time_job'] == 'No']['final_exam_score']
t_stat, p_value = stats.ttest_ind(notas_sin_trabajo, notas_con_trabajo)

print(f'T de student para la prueba:{t_stat:.2f}')
print(f'p-valor: {p_value:.7f}')

T de student para la prueba:4.72
p-valor: 0.0000027


### Resultados de la prueba T de Student
>Con un p-valor inferior a 0.05, rechazamos la hipótesis nula, confirmando que trabajar a tiempo parcial sí tiene un efecto real sobre la nota final

### Estudio del tamaño del efecto
> Confirmar que existe una diferencia en las notas es solo el primer paso. 
>La significancia estadística nos confirma que el efecto existe, pero no cuantifica su magnitud.
>
> Recurriremos al cálculo del tamaño del efecto para evaluar la relevancia práctica que tiene esta diferencia sobre el rendimiento de los alumnos, utilizando la d de Cohen.

In [7]:
import numpy as np
d_cohen = t_stat * np.sqrt(1/len(notas_sin_trabajo) + 1/len(notas_con_trabajo))

print(f"la d de Cohen es: {d_cohen:.2f}")

la d de Cohen es: 0.32


## Conclusiones
> Este estudio tenia por objetivo determinar si el hecho de tener un trabajo a tiempo parcial afecta en la nota del examen final.
>
> Utilizando la prueba T de Student, se ha determiando que compaginar los estudios con un empleo a tiempo parcial penaliza estadísticamente el rendimiento académico
>
>Utilizando la d de Cohen, se ha observado que el impacto real sobre la calificación del alumno es relativamente pequeño. 
>
>Esto sugiere que, en general, los estudiantes logran gestionar su tiempo de forma eficaz, evitando que el trabajo provoque una caída drástica en sus notas finales. Sin embargo, fomentar programas de gestión del tiempo podría ser beneficioso para los estudiantes con trabajo.